In [ ]:
        # import json
        # from typing import Dict, Set, List, Tuple


        # def load_user_follows(path: str) -> Tuple[Set[str], Dict[str, List[str]]]:
        #     """
        #     Load users.json and return:
        #         user_dids: Set[str]
        #         followed: Dict[str, List[str]]
        #             did -> list of author DIDs they follow
        #     """

        #     with open(path, "r", encoding="utf-8") as f:
        #         users: Dict[str, dict] = json.load(f)

        #     # All user DIDs
        #     user_dids: Set[str] = set(users.keys())

        #     # Build followed mapping
        #     followed: Dict[str, List[str]] = {
        #         did: user_data.get("follows_authors", [])
        #         for did, user_data in users.items()
        #     }

        #     return user_dids, followed

In [ ]:
# import json
# from pathlib import Path


# def infer_parent_author_dids_and_save(users: dict, output_path: str) -> None:
#     """
#     1. Infers missing parent_author_did for replies.
#     2. Saves updated users dict to output_path as JSON.

#     Mutates users in place.
#     """

#     def extract_did_from_uri(uri: str | None) -> str | None:
#         if not uri or not isinstance(uri, str):
#             return None
#         if not uri.startswith("at://"):
#             return None
#         try:
#             return uri[5:].split("/", 1)[0]
#         except Exception:
#             return None

#     # -------------------------------
#     # Fix missing parent_author_did
#     # -------------------------------

#     for user_data in users.values():

#         history = user_data.get("history") or []

#         for item in history:

#             if item.get("activity_type") != "reply":
#                 continue

#             if item.get("parent_author_did"):
#                 continue

#             parent_uri = item.get("parent_post_uri")
#             inferred_did = extract_did_from_uri(parent_uri)

#             if inferred_did:
#                 item["parent_author_did"] = inferred_did

#     # -------------------------------
#     # Save to disk
#     # -------------------------------

#     path = Path(output_path)
#     path.parent.mkdir(parents=True, exist_ok=True)

#     with path.open("w", encoding="utf-8") as f:
#         json.dump(users, f, ensure_ascii=False, indent=2)

In [3]:
import json
def get_json(name):
    with open(f"{name}.json", "r", encoding="utf-8") as f:
        return json.load(f)

In [4]:
posts = get_json("../data/raw/posts/postsFinal")
raw_user_data = get_json("../data/raw/users/usersFinal")

In [17]:
def print_basic_stats(users, posts_dict,x=0):
    # -------------------------
    # Authors followed per user
    # -------------------------
    follow_counts = [
        len(u.get("follows_authors", []))
        for u in users.values()
    ]

    followees_count = [u.get("stats").get("follows") for u in users.values()]
    avg = sum(followees_count)/len(followees_count)

    avg_authors_followed = (
        sum(follow_counts) / len(follow_counts)
        if follow_counts else 0
    )
    max_authors_followed = max(follow_counts) if follow_counts else 0
    # -------------------------
    # Reposts per post
    # -------------------------
    repost_counts = [
        len(p.get("stored_reposters", []))
        for p in posts_dict.values()
        if "stored_reposters" in p
    ]
    pos_instances = sum(repost_counts)

    avg_reposts_per_post = (
        sum(repost_counts) / len(repost_counts)
        if repost_counts else 0
    )
    max_reposts_per_post = max(repost_counts) if repost_counts else 0

    posts_with_reposts = sum(c > 0 for c in repost_counts)
    pct_posts_with_reposts = (
        100 * posts_with_reposts / len(posts_dict)
        if repost_counts else 0
    )


    # -------------------------
    # Repost timestamps (from history)
    # -------------------------
    total_reposts = 0
    reposts_with_time = 0

    for u in users.values():
        for h in u.get("history", []):
            if h.get("activity_type") == "repost":
                total_reposts += 1
                if h.get("reposted_at"):
                    reposts_with_time += 1

    pct_reposts_with_time = (
        100 * reposts_with_time / total_reposts
        if total_reposts else 0
    )

    count = sum(
        p.get("repostCount", 0) > x
        for p in posts_dict.values()
    )
    

    num_posts = len(posts_dict)
    post_uris = set(posts_dict.keys())

    # Historical posts excluding overlaps with main collected posts
    historical_uris = set()

    for u in users.values():
        for h in u.get("history") or []:
            uri = h.get("post_uri")
            if uri and uri not in post_uris:
                historical_uris.add(uri)

    num_historical_messages = len(historical_uris)
    total_messages = num_posts + num_historical_messages

    # -------------------------
    # Print results
    # -------------------------
    print("===== BASIC DATASET STATS =====")
    print(f"Users: {len(users)}")
    print(f"Posts: {len(posts_dict)}")
    print(f"Posts with >{x} reposts: {count}/{len(posts_dict)}")
    print()

    print(
        f"Authors followed per user → "
        f"avg: {avg_authors_followed:.2f}, "
        f"max: {max_authors_followed}"
    )
    print(
        f"Avg #followees per user → "
        f"avg: {avg:.2f}, "

    )

    print(
        f"Stored reposters per post → "
        f"avg: {avg_reposts_per_post:.2f}, "
        f"max: {max_reposts_per_post}"
    )

    print(
        f"Posts with ≥1 repost → "
        f"{posts_with_reposts} "
        f"({pct_posts_with_reposts:.2f}%)"
    )


    print(
        f"Reposts with timestamp → "
        f"{reposts_with_time}/{total_reposts} "
        f"({pct_reposts_with_time:.2f}%)"
    )

    print(
        f"Total #positive instances: "
        f"{pos_instances} (from posts)  "
        )
    
    print(
        f"Total number of messages collected: "
        f"{total_messages}"
        )


In [18]:
print_basic_stats(raw_user_data,posts)

===== BASIC DATASET STATS =====
Users: 52925
Posts: 108730
Posts with >0 reposts: 33603/108730

Authors followed per user → avg: 61.84, max: 4284
Avg #followees per user → avg: 2023.48, 
Stored reposters per post → avg: 1.92, max: 3
Posts with ≥1 repost → 32173 (29.59%)
Reposts with timestamp → 1188167/1188167 (100.00%)
Total #positive instances: 61633 (from posts)  
Total number of messages collected: 2006221


In [ ]:
#Ziming had 44,000 pos instances. hence 1:5 dataset was 44,000 x 6

In [7]:
def count_unique_authors(posts_dict):
    """
    Returns the number of unique author DIDs in posts_dict.
    """
    return len({
        (
            p.get("author", {}).get("did")
            or p.get("post", {}).get("author", {}).get("did")
        )
        for p in posts_dict.values()
        if p
    })
num_authors = count_unique_authors(posts)
print(f"Unique authors: {num_authors}")

Unique authors: 29293


In [6]:
def count_reposted_posts_per_hashtag(posts_dict):
    counts = {}

    for post in posts_dict.values():
        source = post.get("hashtag")
        reposters = post.get("reposted_by",[])

        if source and len(reposters) > 0:
            counts[source] = counts.get(source, 0) + 1

    print("------- Pos instances -------")
    for source, count in sorted(counts.items()):
        print(f"{source}: {count}")

count_reposted_posts_per_hashtag(posts)

------- Pos instances -------
AI: 2386
Anime: 4481
BlackHistoryMonth: 4777
Booksky: 3764
Gaza: 3822
ICE: 2968
Pokemon: 4098
Superbowl: 1901
TheTraitors: 1361
Trump: 2615


In [5]:
def count_posts_per_hashtag(posts_dict):
    counts = {}

    for post in posts_dict.values():
        source = post.get("hashtag")

        if source:
            counts[source] = counts.get(source, 0) + 1

    print("------- Stored posts -------")
    for source, count in sorted(counts.items()):
        print(f"{source}: {count}")

count_posts_per_hashtag(posts)

------- Stored posts -------
AI: 11000
Anime: 10697
BlackHistoryMonth: 10935
Booksky: 10935
Gaza: 10973
ICE: 10987
Pokemon: 10989
Superbowl: 10959
TheTraitors: 11000
Trump: 10255
